<a href="https://colab.research.google.com/github/OscarPellicer/extra-attributions/blob/main/notebooks/02_dl_attributions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 2 · Explaining a CNN: attribution methods, sanity checks and evaluation

In this notebook we explain the predictions of an ImageNet CNN (**VGG16**) with [Captum](https://captum.ai) and evaluate the explanations with [Quantus](https://github.com/understandable-machine-intelligence-lab/Quantus).

| # | Section | What you will see |
|---|---------|-------------------|
| 1 | Setup | Model, image, top-5 prediction |
| 2 | Gradient-based methods | Saliency, Guided Backprop, Input×Gradient |
| 3 | Integrated Gradients | Baselines, `n_steps`, completeness |
| 4 | SmoothGrad (`NoiseTunnel`) | Noise level and number of samples |
| 5 | Grad-CAM | Layer choice, class-discriminativeness |
| 6 | Occlusion | Window size and stride |
| 7 | LRP | Propagation rules |
| 8 | Contrastive explanations | *Why hen and not rooster?* |
| 9 | Sanity check | Model parameter randomisation (Adebayo et al., 2018) |
| 10 | Quantus | Faithfulness, robustness and randomisation metrics |
| 11 | *(Optional)* Extras | `TiledOcclusion`, `FusionGrad`, `ContrastiveAttribution` |

**How to work:** run the cells in order (≈ 1-5 min in total on a GPU or Apple-silicon laptop, plus a one-off download of the model weights). Cells with parameters marked `# 🧠 change me` are used in the exercises: **you never need to write new code**, just change the value, re-run the cell, look at the result and answer the question. Then open the *Answer* block to check.

## 1. Setup

In [ ]:
# Install the dependencies (only on Google Colab)
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q captum quantus lovely-tensors git+https://github.com/OscarPellicer/extra-attributions.git
else:
    print('Running locally. If anything is missing, from the root of the extra-attributions repo run:\n'
          '    pip install -r requirements.txt\n    pip install -e .')

In [ ]:
import os
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')  # Apple-silicon: fall back to CPU for unsupported ops

import copy, json, time, urllib.request, warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image
from scipy.stats import spearmanr

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms

from captum.attr import (Saliency, GuidedBackprop, InputXGradient, IntegratedGradients, NoiseTunnel,
                         LayerGradCam, GuidedGradCam, LayerAttribution, Occlusion, LRP)
from captum.attr._utils.lrp_rules import EpsilonRule, GammaRule

import lovely_tensors as lt
lt.monkey_patch()  # nicer printing of tensors
warnings.filterwarnings('ignore', message='.*required_grads has been set automatically.*')
warnings.filterwarnings('ignore', message='.*Setting backward hooks on ReLU activations.*')

# Device: CUDA > Apple MPS > CPU
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print(f'Using {DEVICE=} | torch {torch.__version__}')
torch.manual_seed(0); np.random.seed(0)

We define some helpers: one to fetch the example images (from the local `media/` folder, or from GitHub when running on Colab), and a plotting function in the style of the previous sessions: **green = positive** attribution (evidence *for* the class), **red = negative** (evidence *against*), white = 0. Unless stated otherwise, each map uses its own symmetric colour scale (99.5th percentile of |attribution|), so compare *patterns*, not absolute colours.

In [ ]:
MEDIA_URL = 'https://raw.githubusercontent.com/OscarPellicer/extra-attributions/main/notebooks/media/'

def get_media(fname):
    '''Return a local path to `fname`, downloading it into ./media if needed.'''
    for folder in [Path('media'), Path('notebooks/media')]:
        if (folder / fname).exists():
            return folder / fname
    Path('media').mkdir(exist_ok=True)
    urllib.request.urlretrieve(MEDIA_URL + fname, Path('media') / fname)
    return Path('media') / fname

# Red-white-green colour map for signed attributions
_colors, _values = ['darkred', 'red', 'lightcoral', 'white', 'palegreen', 'green', 'darkgreen'], [0, .15, .4, .5, .6, .9, 1.]
RdGn = LinearSegmentedColormap.from_list('RdGn', list(zip(_values, _colors)), N=256)

def to_2d(attr):
    '''(1,C,H,W) / (C,H,W) / (H,W) attribution -> (H,W) numpy array (mean over channels).'''
    a = torch.as_tensor(attr).detach().float().cpu()
    while a.dim() > 3:
        a = a[0]
    return (a.mean(0) if a.dim() == 3 else a).numpy()

def to_rgb(image):
    return image.detach().cpu().permute(1, 2, 0).clamp(0, 1).numpy()

def plot_attributions(image, attributions=[], attribution_names=[], limits=None, title=None,
                      figsize=3, percentile=99.5):
    '''Plot the input image followed by one signed attribution map per method.'''
    n = 1 + len(attributions)
    fig, axes = plt.subplots(1, n, figsize=(figsize * n, figsize + 0.5), squeeze=False)
    axes = axes[0]
    axes[0].imshow(to_rgb(image)); axes[0].set_title('Input')
    for i, (ax, attr, name) in enumerate(zip(axes[1:], attributions, attribution_names)):
        m = to_2d(attr)
        lim = limits[i] if isinstance(limits, list) else limits
        if lim is None:
            v = max(np.percentile(np.abs(m), percentile), 1e-12)
            lim = (-v, v)
        im = ax.imshow(m, cmap=RdGn, vmin=lim[0], vmax=lim[1])
        ax.set_title(name)
        cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03); cb.ax.tick_params(labelsize=7)
    for ax in axes:
        ax.axis('off')
    if title:
        fig.suptitle(title, fontsize=14)
    plt.tight_layout(); plt.show()

def plot_overlay(image, heatmaps, names, title=None, figsize=3):
    '''Classic Grad-CAM style: non-negative heatmaps overlaid on the image.'''
    n = 1 + len(heatmaps)
    fig, axes = plt.subplots(1, n, figsize=(figsize * n, figsize + 0.5), squeeze=False)
    axes = axes[0]
    axes[0].imshow(to_rgb(image)); axes[0].set_title('Input')
    for ax, h, name in zip(axes[1:], heatmaps, names):
        h = to_2d(h)
        ax.imshow(to_rgb(image)); ax.imshow(h / (h.max() + 1e-12), cmap='jet', alpha=0.5, vmin=0, vmax=1)
        ax.set_title(name)
    for ax in axes:
        ax.axis('off')
    if title:
        fig.suptitle(title, fontsize=14)
    plt.tight_layout(); plt.show()

### Model and image

We use **VGG16** pretrained on ImageNet: it is a plain chain of `Conv2d`/`ReLU`/`MaxPool`/`Linear` layers, which makes it compatible with every method here, including Captum's **LRP** (ResNets/EfficientNets have skip connections or layers that Captum's LRP does not support). The first run downloads ~530 MB of weights.

> We set all `ReLU`s to `inplace=False`: in-place ReLUs break the backward hooks used by Guided Backprop and LRP.

In [ ]:
model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).eval().to(DEVICE)
for m in model.modules():
    if isinstance(m, nn.ReLU):
        m.inplace = False

with open(get_media('imagenet_class_index.json')) as f:
    idx_to_labels = json.load(f)
def idx_to_label(i):
    return idx_to_labels[str(int(i))][1]
label_to_idx = {v[1]: int(k) for k, v in idx_to_labels.items()}

# We resize the whole image to 224x224 (instead of a center crop) so that nothing is cut out
transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

def load_image(fname, crop=None):
    '''Returns (image in [0,1] (3,224,224) on CPU, normalised input batch (1,3,224,224) on DEVICE).'''
    img = Image.open(get_media(fname)).convert('RGB')
    if crop is not None:
        img = img.crop(crop)
    x_img = transform(img)
    return x_img, normalize(x_img)[None].to(DEVICE)

def show_top5(x):
    with torch.no_grad():
        probs = F.softmax(model(x), dim=1)
    top_p, top_i = probs.topk(5)
    for k, (p, i) in enumerate(zip(top_p[0], top_i[0])):
        print(f' {k+1}. {idx_to_label(i):<20s} {p.item()*100:6.2f}%   (class index {i.item()})')
    return top_i[0].tolist()

In [ ]:
IMAGE = 'hen_rooster.jpg'   # 🧠 change me: 'hen_rooster.jpg' or 'swan.jpg'

x_img, x = load_image(IMAGE)
print(x)  # lovely_tensors summary of the normalised input
top5 = show_top5(x)
pred_idx = top5[0]
plt.figure(figsize=(3, 3)); plt.imshow(to_rgb(x_img)); plt.axis('off'); plt.title(IMAGE); plt.show()

The model is torn between **cock** (rooster) and **hen**: both birds are in the picture. Keep this in mind: by default we explain the top-1 class, but the image contains evidence for *both* classes.

## 2. Gradient-based methods: Saliency, Guided Backprop, Input×Gradient

* **Saliency** (vanilla gradient): $\partial f_c(x) / \partial x$. Which pixels, if changed slightly, change the class score the most.
* **Guided Backprop**: like the gradient, but negative gradients are zeroed when passing back through every ReLU. Very crisp maps...
* **Input×Gradient**: $x \odot \partial f_c / \partial x$, a first-order "contribution" of each pixel (equal to LRP-0 for ReLU networks).

In [ ]:
TARGET = pred_idx   # 🧠 change me: e.g. label_to_idx['hen'], label_to_idx['cock'], label_to_idx['goldfish']

attr_sal = Saliency(model).attribute(x, target=TARGET, abs=False)
attr_gbp = GuidedBackprop(model).attribute(x, target=TARGET)
attr_ixg = InputXGradient(model).attribute(x, target=TARGET)

plot_attributions(x_img, [attr_sal, attr_gbp, attr_ixg], ['Saliency', 'Guided Backprop', 'Input×Gradient'],
                  title=f'Target: {idx_to_label(TARGET)}')

### 🧠 Exercise
Change `TARGET` to `label_to_idx['hen']`, then to `label_to_idx['cock']`, and finally to a class that has nothing to do with the image, such as `label_to_idx['goldfish']`. Which of the three methods changes the most with the target class? Which one barely changes?

*Hint:* Look at Guided Backprop in particular. Does it look more like an explanation of a *class* or like an edge detector of the *image*?

<details><summary>Answer</summary>

For *hen* vs. *cock* all three maps look almost the same: these two classes share most of their evidence, and pixel-level gradients cannot tell them apart (we come back to this in §8). For *goldfish*, Saliency and Input×Gradient do change (different speckle pattern and a ~3× smaller colour-bar range), but **Guided Backprop is practically identical for all three classes**, even for *goldfish*: it essentially reconstructs the edges of the image and is (nearly) insensitive to the class and to the upper layers of the network (Nie et al., 2018; Adebayo et al., 2018). A nice-looking map is not necessarily a faithful explanation; we test this formally in the sanity check (§9).

</details>

## 3. Integrated Gradients (IG)

IG integrates the gradient along a straight path from a **baseline** $x'$ to the input $x$:

$$\mathrm{IG}_i(x) = (x_i - x'_i)\int_0^1 \frac{\partial f_c(x' + \alpha (x - x'))}{\partial x_i}\, d\alpha$$

It satisfies **completeness**: $\sum_i \mathrm{IG}_i(x) = f_c(x) - f_c(x')$. Captum approximates the integral with `n_steps` points and returns the *convergence delta* (how far the sum of attributions is from $f_c(x) - f_c(x')$). The baseline encodes "absence of signal", and different choices give different explanations.

In [ ]:
def make_baseline(kind, x_img):
    '''Baselines are built in [0,1] image space and then normalised like the input.'''
    if kind == 'black':
        b = torch.zeros_like(x_img)
    elif kind == 'white':
        b = torch.ones_like(x_img)
    elif kind == 'mean':      # = 0 in normalised space
        b = torch.tensor([0.485, 0.456, 0.406])[:, None, None].expand_as(x_img)
    elif kind == 'blur':
        b = transforms.GaussianBlur(kernel_size=51, sigma=20)(x_img)
    elif kind == 'uniform noise':
        b = torch.rand(x_img.shape, generator=torch.Generator().manual_seed(0))
    else:
        raise ValueError(kind)
    return normalize(b)[None].to(DEVICE)

In [ ]:
TARGET = pred_idx                                  # 🧠 change me
N_STEPS = 32                                       # 🧠 change me: e.g. 2, 8, 32, 128
BASELINES = ['black', 'mean', 'blur', 'uniform noise']   # 🧠 change me: also 'white'

ig = IntegratedGradients(model)
attr_ig_all = {}
for kind in BASELINES:
    baseline = make_baseline(kind, x_img)
    attr, delta = ig.attribute(x, baselines=baseline, target=TARGET, n_steps=N_STEPS,
                               internal_batch_size=16, return_convergence_delta=True)
    with torch.no_grad():
        diff = (model(x)[0, TARGET] - model(baseline)[0, TARGET]).item()
    print(f'{kind:>14s} baseline | sum(IG) = {attr.sum().item():7.3f} | f(x) - f(baseline) = {diff:7.3f} '
          f'| delta = {delta.item():+.4f}')
    attr_ig_all[kind] = attr
attr_ig = attr_ig_all[BASELINES[0]]

plot_attributions(x_img, list(attr_ig_all.values()), [f'IG ({k} baseline)' for k in attr_ig_all],
                  title=f'Target: {idx_to_label(TARGET)}  |  n_steps = {N_STEPS}')

### 🧠 Exercise
Set `N_STEPS = 2`, then `8`, then `128`. How does the convergence `delta` change? Do the maps change visibly?

*Hint:* Completeness holds for the exact integral; `delta` measures the error of the numerical approximation. Captum uses Gauss-Legendre quadrature by default.

<details><summary>Answer</summary>

With 2 steps the deltas are large (several units, i.e. 20-40% of $f(x) - f(x')$ ≈ 20): the attributions do not add up to the output difference. With 32 steps they drop to a few tenths, and with 128 steps to ≲ 0.2. The maps themselves change less than the delta suggests (the overall pattern is already visible with few steps), which is exactly why you should **check the delta rather than trust the picture**. The cost grows linearly with `N_STEPS` (one forward+backward pass per step), so one picks the smallest `N_STEPS` with an acceptable delta (e.g. < 5% of $|f(x) - f(x')|$). The delta also depends on the baseline: paths through more non-linear regions need more steps.

</details>

### 🧠 Exercise
Compare the maps for the `black` and `blur` baselines, focusing on the **black rooster body** and the black background. Then add `'white'` to `BASELINES`. What happens to the dark pixels?

*Hint:* IG is proportional to $(x_i - x'_i)$. What is $x_i - x'_i$ for a black pixel when the baseline is black?

<details><summary>Answer</summary>

With a **black baseline**, every black pixel has $x_i - x'_i = 0$, so it gets **exactly zero attribution** no matter how important it is: the black background and the black body of the rooster are "invisible", and the explanation of *cock* is dominated by the (white) hen! With a **white baseline** the opposite happens: the white hen almost disappears and the dark rooster lights up. Blurred / mean baselines are in between. There is no neutral baseline: it defines the question "important **compared to what**?". Averaging over several baselines (e.g. Expected Gradients / GradientSHAP) or using a blurred baseline are common mitigations.

</details>

## 4. SmoothGrad / `NoiseTunnel`

Gradients of deep networks are very noisy (they fluctuate sharply at small scales). **SmoothGrad** averages the attribution over `nt_samples` copies of the input with Gaussian noise of standard deviation `stdevs` added. In Captum, `NoiseTunnel` wraps *any* attribution method. Following the SmoothGrad paper, we express the noise as a fraction of the input range (they recommend 10-20%), and we show Saliency as **absolute** values (|gradient|), as in the paper: signed gradients flip sign from pixel to pixel and their average is hard to read.

In [ ]:
TARGET = pred_idx      # 🧠 change me
NT_SAMPLES = 16        # 🧠 change me: e.g. 4, 16, 64
NOISE_LEVEL = 0.15     # 🧠 change me: e.g. 0.0, 0.05, 0.15, 0.5  (std as a fraction of the input range)

stdev = NOISE_LEVEL * (x.max() - x.min()).item()
nt_kwargs = dict(nt_type='smoothgrad', nt_samples=NT_SAMPLES, nt_samples_batch_size=8, stdevs=stdev, target=TARGET)

t0 = time.perf_counter()
attr_sal_sg = NoiseTunnel(Saliency(model)).attribute(x, abs=True, **nt_kwargs)
attr_ig_sg = NoiseTunnel(IntegratedGradients(model)).attribute(x, n_steps=8, internal_batch_size=16, **nt_kwargs)
print(f'Time: {time.perf_counter() - t0:.1f}s  (noise std = {stdev:.2f} in normalised units)')

attr_sal_abs = Saliency(model).attribute(x, target=TARGET, abs=True)
attr_ig = IntegratedGradients(model).attribute(x, target=TARGET, n_steps=32, internal_batch_size=16)
plot_attributions(x_img, [attr_sal_abs, attr_sal_sg, attr_ig, attr_ig_sg],
                  ['|Saliency|', '|Saliency| + SG', 'IG', 'IG + SG'],
                  title=f'Target: {idx_to_label(TARGET)}  |  noise = {NOISE_LEVEL:.0%}, samples = {NT_SAMPLES}')

### 🧠 Exercise
Try `NOISE_LEVEL = 0.0`, `0.05` and `0.5`. What happens at both extremes?

*Hint:* With zero noise, all samples are identical. With huge noise, what is left of the image?

<details><summary>Answer</summary>

With `NOISE_LEVEL = 0` SmoothGrad is identical to the base method (it averages identical maps). With 5% the speckle is already reduced and |Saliency| + SG starts to concentrate on the heads. Around 15% the |Saliency| map is smooth and clearly focused on the hen's and rooster's heads, and IG + SG shows clean contours. With 50% the noisy inputs no longer resemble the image: the |Saliency| map turns into a blurry, global shape (mostly the outline between the birds and the background) and IG + SG becomes pure noise. SmoothGrad trades **fidelity to the exact input** for **visual coherence/robustness**; the noise level is a hyperparameter you have to justify.

</details>

### 🧠 Exercise
Keep `NOISE_LEVEL = 0.15` and compare `NT_SAMPLES = 4` and `NT_SAMPLES = 64`. Look at the time printed and at the maps.

*Hint:* SmoothGrad is a Monte Carlo estimate of an expectation.

<details><summary>Answer</summary>

With 4 samples the average is itself noisy (high Monte Carlo variance), with 64 it converges to a smooth map. The cost is linear in `NT_SAMPLES` (and multiplied by `n_steps` for IG + SG!), as the printed time shows. In practice 15-50 samples are typical.

</details>

## 5. Grad-CAM

Grad-CAM works at the level of a **convolutional layer** $A$ (channels $A^k$): it weights each channel by its spatially averaged gradient, $\alpha_k^c = \frac{1}{Z}\sum_{ij} \partial f_c / \partial A^k_{ij}$, and keeps the positive part: $\mathrm{ReLU}(\sum_k \alpha_k^c A^k)$. The result has the **resolution of the layer** (14×14 for the last conv layer of VGG16), so we upsample it to the input size with `LayerAttribution.interpolate`. **Guided Grad-CAM** multiplies the upsampled Grad-CAM by Guided Backprop to get a high-resolution map.

The indices of `model.features` in VGG16 are: conv/ReLU blocks at 0-3 (224×224), 5-8 (112×112), 10-15 (56×56), 17-22 (28×28), 24-29 (14×14), with max-pooling at 4, 9, 16, 23, 30.

In [ ]:
TARGET = pred_idx      # 🧠 change me
GRADCAM_LAYER = 29     # 🧠 change me: index in model.features, e.g. 30 (7x7), 29 (14x14), 22 (28x28), 15 (56x56)

layer = model.features[GRADCAM_LAYER]
attr_gc = LayerGradCam(model, layer).attribute(x, target=TARGET, relu_attributions=True)
attr_gc_up = LayerAttribution.interpolate(attr_gc, x.shape[-2:], interpolate_mode='bilinear')
attr_ggc = GuidedGradCam(model, layer).attribute(x, target=TARGET)
print(f'Layer: model.features[{GRADCAM_LAYER}] = {layer} | Grad-CAM resolution: {tuple(attr_gc.shape[-2:])}')

plot_attributions(x_img, [attr_gc, attr_gc_up, attr_ggc], ['Grad-CAM (raw)', 'Grad-CAM (upsampled)', 'Guided Grad-CAM'],
                  title=f'Target: {idx_to_label(TARGET)}')

Grad-CAM is usually shown as an overlay. Let us compare the explanations for the **top-2 predicted classes** (*cock* and *hen*). Grad-CAM is often described as "class-discriminative"... is it here?

In [ ]:
heatmaps = []
for c in top5[:2]:
    gc = LayerGradCam(model, model.features[GRADCAM_LAYER]).attribute(x, target=c, relu_attributions=True)
    heatmaps.append(LayerAttribution.interpolate(gc, x.shape[-2:], interpolate_mode='bilinear'))
plot_overlay(x_img, heatmaps, [f'Grad-CAM: {idx_to_label(c)}' for c in top5[:2]])

### 🧠 Exercise
First, look at the default overlays: does Grad-CAM distinguish *cock* from *hen*? Then change `GRADCAM_LAYER` to `30`, `22` and `15` and re-run the two cells above. How does the resolution change, and does any layer separate the two classes?

*Hint:* Deeper layers have lower resolution but more semantic features. Compare the *cock* and *hen* overlays for each layer.

<details><summary>Answer</summary>

Surprisingly, **at the default layer (29) the *cock* and *hen* maps are almost identical**: both point at the hen's head. Grad-CAM tells us *where the "chicken" evidence is*, not what separates a hen from a rooster. At layer 30 (7×7) the maps are even coarser and still identical. At 22 (28×28) and 15 (56×56) the resolution improves and the maps pick out parts (combs, eyes, the rooster's neck feathers, and also some spurious activity on the wooden fence), but they remain very similar for both classes and become noisier: intermediate features are generic (feathers, edges) and averaging gradients over channels is a cruder approximation far from the output. Grad-CAM is normally applied to the last convolutional layer; to separate two similar classes we need a **contrastive** explanation (§8).

</details>

### 🧠 Exercise
Set `TARGET = label_to_idx['goldfish']` with `GRADCAM_LAYER = 29`. Compare the result with Guided Backprop for *goldfish* in §2.

*Hint:* Grad-CAM keeps only the positive part (ReLU). What does it mean if almost nothing is positive?

<details><summary>Answer</summary>

Check the colour bars: the *goldfish* Grad-CAM is ~6× weaker than for *cock* and patchy, i.e. there is little positive evidence for *goldfish*. What little there is sits on the **red combs and wattles**, the closest thing to an orange goldfish in the image, and Guided Grad-CAM accordingly keeps only the edges of the combs. Guided Backprop alone, instead, still shows a sharp picture of both birds because it hardly depends on the target: **Guided Grad-CAM** inherits its class-sensitivity from the Grad-CAM factor, which is why it was proposed.

</details>

## 6. Occlusion

A perturbation method: slide a patch of size `WINDOW` with step `STRIDE` over the image, replace it with a baseline (here 0 = mean colour after normalisation), and record how much the class score drops. Model-agnostic and easy to interpret, but the cost is one forward pass per patch position.

In [ ]:
TARGET = pred_idx   # 🧠 change me
WINDOW = 32         # 🧠 change me: e.g. 16, 32, 64
STRIDE = 16         # 🧠 change me: e.g. 8, 16, 32

n_evals = ((224 - WINDOW) // STRIDE + 1) ** 2
t0 = time.perf_counter()
attr_occ = Occlusion(model).attribute(x, target=TARGET, sliding_window_shapes=(3, WINDOW, WINDOW),
                                      strides=(3, STRIDE, STRIDE), baselines=0, perturbations_per_eval=32)
print(f'{n_evals} forward passes in {time.perf_counter() - t0:.1f}s')
plot_attributions(x_img, [attr_occ], [f'Occlusion (w={WINDOW}, s={STRIDE})'], title=f'Target: {idx_to_label(TARGET)}')

### 🧠 Exercise
Try `(WINDOW, STRIDE) = (64, 32)`, `(32, 16)` and `(16, 8)`. How do the map and the computing time change?

*Hint:* The number of forward passes is ≈ $((224 - w)/s + 1)^2$. What can a 16×16 grey patch hide that a 64×64 patch can, and vice versa?

<details><summary>Answer</summary>

Large windows give coarse, blocky but stable maps (hiding a whole head clearly matters); they cannot localise small parts. Small windows give finer maps but (i) cost many more passes (729 for 16/8 vs 49 for 64/32) and (ii) a small patch often does not change the prediction because the rest of the object still provides enough evidence (**redundancy**), so the map can become weak or noisy. The window size should match the scale of the features you expect to matter. (The optional `TiledOcclusion` in §11 increases the resolution without shrinking the window.)

</details>

### 🧠 Exercise
With the default parameters, set `TARGET` to `label_to_idx['hen']` and then `label_to_idx['cock']`. Do you see red (negative) regions? What do they mean here?

*Hint:* Occlusion attribution = score(original) − score(occluded). When is it negative?

<details><summary>Answer</summary>

Yes. For *hen*, the hen's body is strongly green, while the rooster's head and comb are red: hiding them *increases* the hen score (they are evidence for *cock*). For *cock*, both birds are mostly green (the model uses chicken features from both). In both cases the black background at the top is red: replacing black with a grey patch increases the score, a reminder that the occlusion "baseline" creates unnatural images and that its effect is not always "removing information". Occlusion directly measures the effect of a perturbation, so it naturally reveals *competing* evidence, which we exploit with contrastive explanations (§8).

</details>

## 7. Layer-wise Relevance Propagation (LRP)

LRP redistributes the output score backwards, layer by layer, conserving the total relevance. Different **rules** can be used at each layer (Montavon et al., 2019):

* **LRP-0 / LRP-ε**: $R_j = \sum_k \frac{a_j w_{jk}}{\epsilon + \sum_j a_j w_{jk}} R_k$. ε absorbs weak/contradictory contributions (less noise). With ε→0 this is LRP-0, which for ReLU networks equals Input×Gradient.
* **LRP-γ**: favours positive contributions ($w + \gamma w^+$), giving more stable maps in the lower layers.

A typical **composite** strategy: LRP-0 for the dense layers, LRP-ε for the upper convolutions and LRP-γ for the lower ones (the very first layer, which sees pixels that can be negative, would ideally use the $z^\mathcal{B}$ rule; Captum does not provide it, so we use LRP-ε there). In Captum, a rule is set by assigning `module.rule`.

In [ ]:
TARGET = pred_idx       # 🧠 change me
USE_COMPOSITE = True    # 🧠 change me: False -> same LRP-ε rule for every layer
LRP_EPSILON = 0.25      # 🧠 change me: e.g. 1e-9 (= LRP-0), 0.25, 10
LRP_GAMMA = 0.25        # 🧠 change me: e.g. 0.0, 0.25, 1.0

def set_lrp_rules(model, epsilon=0.25, gamma=0.25, composite=True):
    layers = [m for m in model.modules() if isinstance(m, (nn.Conv2d, nn.Linear))]
    n_conv = sum(isinstance(m, nn.Conv2d) for m in layers)
    for i, m in enumerate(layers):
        if not composite:
            m.rule = EpsilonRule(epsilon=epsilon)
        elif isinstance(m, nn.Linear):
            m.rule = EpsilonRule(epsilon=1e-9)           # LRP-0 for dense layers
        elif i == 0:
            m.rule = EpsilonRule(epsilon=epsilon)        # first layer: pixels can be negative, γ is not appropriate
        elif i <= n_conv // 3:
            m.rule = GammaRule(gamma=gamma)              # LRP-γ for the lower convs
        else:
            m.rule = EpsilonRule(epsilon=epsilon)        # LRP-ε for the upper convs
    return model

lrp_model = set_lrp_rules(copy.deepcopy(model), LRP_EPSILON, LRP_GAMMA, USE_COMPOSITE)  # copy: rules are attached to modules
attr_lrp = LRP(lrp_model).attribute(x, target=TARGET)
print(f'sum of relevance = {attr_lrp.sum().item():.3f}  |  logit = {model(x)[0, TARGET].item():.3f}')

attr_ixg = InputXGradient(model).attribute(x, target=TARGET)
rule_name = 'composite' if USE_COMPOSITE else f'ε = {LRP_EPSILON:g}'
plot_attributions(x_img, [attr_ixg, attr_lrp], ['Input×Gradient', f'LRP ({rule_name})'],
                  title=f'Target: {idx_to_label(TARGET)}')

### 🧠 Exercise
Set `USE_COMPOSITE = False` and `LRP_EPSILON = 1e-9` (i.e. LRP-0 everywhere). Compare the LRP map with Input×Gradient. Then try `LRP_EPSILON = 10`.

*Hint:* For ReLU networks, LRP-0 is mathematically equivalent to Input×Gradient. Also compare the printed *sum of relevance* with the logit.

<details><summary>Answer</summary>

With LRP-0 everywhere the LRP map is **identical to Input×Gradient**, noise included (correlation 1.0): for ReLU networks LRP-0 = Gradient×Input. With a large ε (10) only the strongest, mostly positive contributions survive: a very sparse map on the heads and combs, but the total relevance collapses (≈ 0.001 instead of ≈ 20): it has been "absorbed" by ε, so conservation no longer holds. The **composite** rule is a compromise: much less noisy than LRP-0, and it highlights the rooster's comb and head for *cock*. LRP is a family of methods whose behaviour depends heavily on the chosen rules.

</details>

### 🧠 Exercise
With `USE_COMPOSITE = True`, compare `LRP_GAMMA = 0.0` and `LRP_GAMMA = 1.0`.

*Hint:* γ controls how much positive contributions are favoured in the lower layers. With γ = 0 the γ-rule reduces to LRP-0.

<details><summary>Answer</summary>

With γ = 0 the lower layers use LRP-0 and the map looks like Input×Gradient again: dense, high-frequency red/green noise over the hen. Increasing γ makes the lower layers favour positive contributions: with γ = 0.25 and especially γ = 1 the noise disappears and the map becomes sparse and contour-like, following the outlines, heads and combs of the birds. Too large γ tends towards "positive-only" maps (like LRP-α1β0) that lose negative evidence.

</details>

### All methods side by side

A summary for the current `TARGET` (re-run after changing it, e.g. to `label_to_idx['hen']`):

In [ ]:
TARGET = pred_idx   # 🧠 change me

gallery = {
    'Saliency': Saliency(model).attribute(x, target=TARGET, abs=False),
    'Guided BP': GuidedBackprop(model).attribute(x, target=TARGET),
    'Input×Grad': InputXGradient(model).attribute(x, target=TARGET),
    'IG (black)': IntegratedGradients(model).attribute(x, target=TARGET, n_steps=32, internal_batch_size=16),
    '|Saliency| + SG': NoiseTunnel(Saliency(model)).attribute(x, nt_type='smoothgrad', nt_samples=16, nt_samples_batch_size=8,
                                                              stdevs=0.15 * (x.max() - x.min()).item(), target=TARGET, abs=True),
    'Grad-CAM': LayerAttribution.interpolate(LayerGradCam(model, model.features[29]).attribute(x, target=TARGET, relu_attributions=True),
                                             x.shape[-2:], interpolate_mode='bilinear'),
    'Guided Grad-CAM': GuidedGradCam(model, model.features[29]).attribute(x, target=TARGET),
    'Occlusion': Occlusion(model).attribute(x, target=TARGET, sliding_window_shapes=(3, 32, 32), strides=(3, 16, 16),
                                            baselines=0, perturbations_per_eval=32),
    'LRP (composite)': LRP(set_lrp_rules(copy.deepcopy(model))).attribute(x, target=TARGET),
}
plot_attributions(x_img, list(gallery.values()), list(gallery.keys()), title=f'Target: {idx_to_label(TARGET)}', figsize=2.3)

### 🧠 Exercise
Go back to §1, set `IMAGE = 'swan.jpg'`, re-run that cell and then the gallery cell above. The model predicts *black_swan* for a **white** mute swan (ImageNet has no "mute swan" class). Which parts of the image do Grad-CAM and Occlusion consider evidence for *black_swan*? (Set `IMAGE` back to `'hen_rooster.jpg'` afterwards.)

*Hint:* Look at where Grad-CAM is strongest and whether the white adult swan is highlighted at all.

<details><summary>Answer</summary>

Grad-CAM is strongest on the **grey cygnets** and on the **water** around them, and much weaker on the white adult swan; Occlusion is positive over most of the image (water included). The model's "black swan" evidence comes largely from the dark chicks and the aquatic context, not from the white swan itself, a plausible explanation of the error (the label is wrong for the main object, but the model found something dark and swan-like). Gradient maps (Saliency, Guided BP, LRP) highlight swan and cygnet contours, which says much less about *why* the class was chosen.

</details>

## 8. Contrastive explanations: *why hen and not rooster?*

The image contains a hen **and** a rooster, and the model hesitates between *hen* and *cock*. Explaining $f_\text{hen}(x)$ alone mixes two kinds of evidence: what makes it a **bird/chicken** (shared by both classes) and what makes it a **hen rather than a rooster**. Humans usually ask contrastive questions (*"why P rather than Q?"*, Miller 2019).

A simple way to answer them: explain the **difference of logits** $f_A(x) - f_B(x)$ instead of $f_A(x)$. We wrap the model so that its single output is that difference, and then use any Captum method on the wrapped model with `target=0`.

In [ ]:
class LogitDifference(nn.Module):
    '''Wraps a classifier so that its single output is logit[class_a] - logit[class_b].'''
    def __init__(self, model, class_a, class_b):
        super().__init__()
        self.model, self.class_a, self.class_b = model, class_a, class_b
    def forward(self, x):
        logits = self.model(x)
        return logits[:, [self.class_a]] - logits[:, [self.class_b]]

In [ ]:
CLASS_A = 'hen'                       # 🧠 change me
CLASS_B = 'cock'                      # 🧠 change me: e.g. 'cock', 'partridge', 'goldfish'
CONTRAST_IMAGE = 'hen_rooster.jpg'    # 🧠 change me

xc_img, xc = load_image(CONTRAST_IMAGE)
a, b = label_to_idx[CLASS_A], label_to_idx[CLASS_B]
with torch.no_grad():
    logits = model(xc)[0]
print(f'logit[{CLASS_A}] = {logits[a]:.2f}, logit[{CLASS_B}] = {logits[b]:.2f}, difference = {logits[a] - logits[b]:.2f}')

models_to_explain = {CLASS_A: (model, a), CLASS_B: (model, b),
                     f'{CLASS_A} − {CLASS_B}': (LogitDifference(model, a, b), 0),
                     f'{CLASS_B} − {CLASS_A}': (LogitDifference(model, b, a), 0)}
occ_maps, gc_maps = {}, {}
for name, (f, t) in models_to_explain.items():
    gc = LayerGradCam(f, model.features[29]).attribute(xc, target=t, relu_attributions=True)
    gc_maps[name] = LayerAttribution.interpolate(gc, xc.shape[-2:], interpolate_mode='bilinear')
    occ_maps[name] = Occlusion(f).attribute(xc, target=t, sliding_window_shapes=(3, 32, 32), strides=(3, 16, 16),
                                            baselines=0, perturbations_per_eval=32)

names = list(models_to_explain)
plot_overlay(xc_img, [gc_maps[n] for n in names], [f'Grad-CAM: {n}' for n in names], title='Grad-CAM')
plot_attributions(xc_img, [occ_maps[n] for n in names[:3]], [f'Occlusion: {n}' for n in names[:3]],
                  title='Occlusion (w=32, s=16)')

Note that Occlusion (like IG, Saliency or Input×Gradient) is **linear in the explained function**: Occlusion(hen − cock) is exactly Occlusion(hen) − Occlusion(cock), so for these methods the contrastive map is simply the difference of the two maps. For Grad-CAM (with its ReLU and channel weighting) this is not the case, and explaining the logit difference directly is the principled choice.

### 🧠 Exercise
Compare the single-class maps (*hen*, *cock*) with the contrastive ones (*hen − cock*, *cock − hen*). Which parts of the image are relevant for both classes and disappear in the contrastive map? Which features distinguish a hen from a rooster according to the model?

*Hint:* Which bird (and which part of it) does each contrastive Grad-CAM point to? Where is the contrastive Occlusion map green, and where red?

<details><summary>Answer</summary>

The single-class Grad-CAMs for *hen* and *cock* are practically identical (both on the hen's head, as we saw in §5), and the single-class Occlusion maps both highlight the two birds: chicken-like features support both classes. In the contrastive explanations the shared evidence cancels out: Grad-CAM for *hen − cock* covers the **white body of the hen**, while *cock − hen* points at the **rooster's large red comb**. The contrastive Occlusion map agrees: green on the hen, red on the rooster's head and comb. So, according to the model, white plumage says "hen" and a big comb says "rooster", which is the human answer too. Contrastive explanations answer the question the user actually has and remove evidence that is irrelevant to the decision between two similar classes.

</details>

### 🧠 Exercise
Keep `CLASS_A = 'hen'` and set `CLASS_B = 'goldfish'` (a class that is not in the image). How does the *hen − goldfish* map compare with the plain *hen* map? Why?

*Hint:* If the logit of class B is almost flat with respect to the input, what does its gradient contribute?

<details><summary>Answer</summary>

The *hen − goldfish* maps are essentially the plain *hen* maps (same Grad-CAM on the hen's head, same Occlusion pattern): the goldfish logit is low and varies little compared with the hen logit, so subtracting it changes almost nothing and the shared "chicken" evidence remains. Contrastive explanations are most informative when the contrast class is a **plausible alternative** (the runner-up prediction, or a class a user might confuse it with).

</details>

## 9. Sanity check: model parameter randomisation (Adebayo et al., 2018)

If an explanation is faithful to the model, it **must change when the model changes**. Adebayo et al. proposed *cascading randomisation*: re-initialise the weights layer by layer, **from the top (output) to the bottom (input)**, and compare each explanation with the one from the trained model. A method whose maps stay the same after the network has been destroyed is explaining the *image* (e.g. edges), not the *model*.

We compute four methods at each step and measure similarity with the **Spearman rank correlation** of the absolute attributions (summed over colour channels), as in the paper. To keep it cheap, IG uses only 16 steps.

In [ ]:
SANITY_METHODS = ['Saliency', 'Guided Backprop', 'Integrated Gradients', 'Grad-CAM']

def explain(method, model, x, target):
    '''Common interface used in the sanity check and in the Quantus evaluation. Returns an (N,C,H,W) tensor.'''
    if method == 'Saliency':
        return Saliency(model).attribute(x, target=target, abs=False)
    if method == 'Guided Backprop':
        return GuidedBackprop(model).attribute(x, target=target)
    if method == 'Input×Gradient':
        return InputXGradient(model).attribute(x, target=target)
    if method == 'Integrated Gradients':
        return IntegratedGradients(model).attribute(x, target=target, n_steps=16, internal_batch_size=16)
    if method == 'SmoothGrad':
        return NoiseTunnel(Saliency(model)).attribute(x, nt_type='smoothgrad', nt_samples=8, nt_samples_batch_size=8,
                                                      stdevs=0.15 * (x.max() - x.min()).item(), target=target, abs=False)
    if method == 'Grad-CAM':
        gc = LayerGradCam(model, model.features[29]).attribute(x, target=target, relu_attributions=True)
        return LayerAttribution.interpolate(gc, x.shape[-2:], interpolate_mode='bilinear')
    raise ValueError(method)

# Names of the 16 weight layers of VGG16, from input to output
vgg_layers = [name for name, m in model.named_modules() if isinstance(m, (nn.Conv2d, nn.Linear))]
block_sizes = [2, 2, 3, 3, 3]
pretty = [f'conv{b+1}_{i+1}' for b, n in enumerate(block_sizes) for i in range(n)] + ['fc6', 'fc7', 'fc8']
layer_names = dict(zip(vgg_layers, pretty))
print(' -> '.join(pretty))

In [ ]:
RANDOMISATION = 'cascading'   # 🧠 change me: 'cascading' (top-down, cumulative) or 'independent' (one layer at a time)
N_LAYERS = 16                 # 🧠 change me: how many layers (counting from the output) to randomise, 1..16
ABS = True                    # 🧠 change me: compare |attributions| (as in the paper) or signed attributions
TARGET = pred_idx

def attribution_map(method, m):
    a = explain(method, m, x, TARGET).detach().sum(1)[0]   # sum over colour channels -> (H, W)
    return (a.abs() if ABS else a).cpu().numpy()

def similarity(a, b):
    if a.std() == 0 or b.std() == 0:   # e.g. an all-zero Grad-CAM map
        return np.nan
    return spearmanr(a.ravel(), b.ravel()).statistic

torch.manual_seed(0)
t0 = time.perf_counter()
stages = ['original'] + [layer_names[n] for n in vgg_layers[::-1][:N_LAYERS]]
maps = {meth: [attribution_map(meth, model)] for meth in SANITY_METHODS}
rand_model = copy.deepcopy(model)
for layer in vgg_layers[::-1][:N_LAYERS]:
    if RANDOMISATION == 'independent':
        rand_model = copy.deepcopy(model)                 # start again from the trained model
    dict(rand_model.named_modules())[layer].reset_parameters()   # re-initialise this layer's weights
    for meth in SANITY_METHODS:
        maps[meth].append(attribution_map(meth, rand_model))
scores = {meth: [similarity(maps[meth][0], m) for m in maps[meth]] for meth in SANITY_METHODS}
print(f'Done in {time.perf_counter() - t0:.1f}s')

In [ ]:
# Maps at some of the randomisation stages (each map normalised to its own range)
show = [s for s in ['original', 'fc8', 'fc7', 'fc6', 'conv5_3', 'conv5_1', 'conv4_1', 'conv3_1', 'conv2_1', 'conv1_1'] if s in stages]
fig, axes = plt.subplots(len(SANITY_METHODS), len(show), figsize=(1.7 * len(show), 1.8 * len(SANITY_METHODS)), squeeze=False)
for r, meth in enumerate(SANITY_METHODS):
    for c, s in enumerate(show):
        m = maps[meth][stages.index(s)]
        ax = axes[r, c]
        if ABS:
            ax.imshow(m, cmap='gray', vmin=0, vmax=max(np.percentile(m, 99.5), 1e-12))
        else:
            v = max(np.percentile(np.abs(m), 99.5), 1e-12); ax.imshow(m, cmap=RdGn, vmin=-v, vmax=v)
        ax.set_xticks([]); ax.set_yticks([])
        if r == 0: ax.set_title(s if s == 'original' else (f'→ {s}' if RANDOMISATION == 'cascading' else f'only {s}'), fontsize=9)
        if c == 0: ax.set_ylabel(meth, fontsize=9)
fig.suptitle(f'{RANDOMISATION.capitalize()} randomisation of VGG16 weights (from the top)')
plt.tight_layout(); plt.show()

# Similarity to the original explanation
plt.figure(figsize=(9, 3.5))
for meth in SANITY_METHODS:
    plt.plot(range(len(stages)), scores[meth], 'o-', label=meth)
plt.xticks(range(len(stages)), stages, rotation=60)
plt.axhline(0, color='grey', lw=0.8)
plt.ylabel('Spearman ρ vs. original'); plt.ylim(-0.3, 1.05)
plt.xlabel(f'{RANDOMISATION} randomisation up to layer →')
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

### 🧠 Exercise
Look at the plots with the default settings. Which method **fails** the sanity check (its maps stay similar to the original even after most of the network has been randomised)? Which one passes most clearly?

*Hint:* A method *passes* if the similarity drops quickly (ideally already after randomising `fc8`, the layer that encodes the classes).

<details><summary>Answer</summary>

**Guided Backprop fails**: its Spearman correlation stays ≈ 1 until the conv4 layers and ≈ 0.75 even when only `conv1_1` is left; the maps still show the bird contours because GBP mostly reconstructs the input. **Grad-CAM** passes most clearly: randomising `fc8` alone already drops its correlation to ≈ 0.45 and then it fluctuates around 0-0.5 (the maps become random blobs). **Saliency** and **IG** drop to ≈ 0.5-0.7 after `fc8` and then decay slowly: part of the structure (where the birds are) persists in the absolute values, and IG additionally multiplies by the input $x - x'$, which keeps image structure in the map.

</details>

### 🧠 Exercise
Set `RANDOMISATION = 'independent'` (randomise one layer at a time, keeping the others trained). Which layers matter most for each method? Then set `ABS = False` back with `'cascading'`: do the conclusions change?

*Hint:* In the independent test, a big drop at layer L means the explanation depends on the weights of layer L.

<details><summary>Answer</summary>

Independent randomisation shows that **every** single layer matters for Saliency, IG and Grad-CAM (randomising e.g. `conv4_3` alone drops Saliency to ≈ 0), whereas Guided Backprop stays at ρ ≈ 1 for the classifier and conv5 layers and only reacts to some middle/lower conv layers. With `ABS = False` (signed attributions, cascading) Saliency and IG drop to **≈ 0 right after `fc7`**: the *sign* of the attribution is destroyed by randomisation, while the *magnitude* pattern partly survives; Guided Backprop stays ≈ 1 even with signs. Adebayo et al. noted that absolute values can make random-model explanations look deceptively similar, so the similarity measure is itself a methodological choice (as is using a single image: in practice, average over many).

</details>

### 🧠 Exercise
Set `N_LAYERS = 3` (randomise only the classifier `fc8, fc7, fc6`). Look at the Guided Backprop maps: would a user be able to tell that the classifier is now completely random?

*Hint:* After randomising fc6-fc8 the model's predictions are meaningless.

<details><summary>Answer</summary>

No: the Guided Backprop maps after randomising the whole classifier are visually almost indistinguishable from the original ones. A user would happily accept them as an explanation of a model that is now random. This is the key message of Adebayo et al.: **visual appeal is not evidence of faithfulness**, and why randomisation tests are part of attribution evaluation.

</details>

## 10. Quantitative evaluation with Quantus

Instead of eyeballing maps, [Quantus](https://github.com/understandable-machine-intelligence-lab/Quantus) provides metrics for several properties. We evaluate three methods on 4 inputs (the full image, crops with only the hen / only the rooster, and the swan) with one metric per family:

| Metric | Property | Idea | Better |
|---|---|---|---|
| `FaithfulnessCorrelation` (Bhatt et al., 2020) | Faithfulness | Perturb random pixel subsets; correlate the sum of their attributions with the drop of the output | ↑ higher |
| `MaxSensitivity` (Yeh et al., 2019) | Robustness | Max. change of the explanation under small input noise | ↓ lower |
| `RandomLogit` (Sixt et al., 2020) | Randomisation | Similarity (SSIM) between the explanation of the predicted class and of a random class | ↓ lower |

Sample counts are deliberately tiny so that it runs in about a minute: **the numbers are noisy** (that is part of the lesson).

In [ ]:
import quantus
import pandas as pd
print(f'Quantus {quantus.__version__}')

METHODS_TO_EVALUATE = ['Saliency', 'Integrated Gradients', 'Grad-CAM']   # 🧠 change me: add 'Guided Backprop', 'SmoothGrad', 'Input×Gradient'
NR_RUNS = 20                 # 🧠 change me: FaithfulnessCorrelation runs per image (Quantus default: 100)
NR_SAMPLES = 4               # 🧠 change me: MaxSensitivity noise samples per image (Quantus default: 200)
PERTURB_BASELINE = 'black'   # 🧠 change me: 'black', 'mean', 'uniform'
QDEVICE = str(DEVICE)       # if Quantus fails on your GPU/MPS, set QDEVICE = 'cpu' (slower)

# Evaluation data: (file, crop box)
eval_images = [('hen_rooster.jpg', None), ('hen_rooster.jpg', (0, 60, 520, 662)),
               ('hen_rooster.jpg', (440, 0, 1000, 662)), ('swan.jpg', None)]
x_batch = torch.cat([load_image(f, crop)[1] for f, crop in eval_images])
with torch.no_grad():
    y_batch = model(x_batch).argmax(1)
print('Predicted:', [idx_to_label(i) for i in y_batch])

def quantus_explain(model, inputs, targets, method='Saliency', **kwargs):
    '''Quantus calls this with numpy arrays; we return channel-summed attributions (N,1,H,W).'''
    dev = next(model.parameters()).device
    x_t = torch.as_tensor(inputs, dtype=torch.float32, device=dev)
    t_t = torch.as_tensor(targets, device=dev).long()
    return explain(method, model, x_t, t_t).sum(1, keepdim=True).detach().cpu().numpy()

In [ ]:
metrics = {
    'FaithfulnessCorrelation': quantus.FaithfulnessCorrelation(
        nr_runs=NR_RUNS, subset_size=224 * 14, perturb_baseline=PERTURB_BASELINE,
        similarity_func=quantus.similarity_func.correlation_pearson,
        abs=False, normalise=True, return_aggregate=False, disable_warnings=True),
    'MaxSensitivity': quantus.MaxSensitivity(
        nr_samples=NR_SAMPLES, lower_bound=0.1,   # uniform noise in [-0.1, 0.1] (default batched perturb_func)
        norm_numerator=quantus.norm_func.fro_norm, norm_denominator=quantus.norm_func.fro_norm,
        similarity_func=quantus.similarity_func.difference,
        abs=False, normalise=True, return_aggregate=False, disable_warnings=True),
    'RandomLogit': quantus.RandomLogit(
        num_classes=1000, similarity_func=quantus.similarity_func.ssim,
        abs=False, normalise=True, return_aggregate=False, disable_warnings=True),
}

qmodel = copy.deepcopy(model).to(QDEVICE)
xq, yq = x_batch.to(QDEVICE), y_batch.to(QDEVICE)
results = {}
t0 = time.perf_counter()
for method in METHODS_TO_EVALUATE:
    a_batch = quantus_explain(qmodel, xq, yq, method=method)
    for metric_name, metric in metrics.items():
        scores_q = metric(model=qmodel, x_batch=xq.cpu().numpy(), y_batch=yq.cpu().numpy(), a_batch=a_batch,
                          explain_func=quantus_explain, explain_func_kwargs={'method': method}, device=QDEVICE)
        results[(method, metric_name)] = np.array(scores_q, dtype=float)
    print(f'{method:<22s} done ({time.perf_counter() - t0:.0f}s)')

In [ ]:
directions = {name: ('↑' if m.score_direction.name == 'HIGHER' else '↓') for name, m in metrics.items()}
df = pd.DataFrame({f'{metric} {directions[metric]}': {meth: results[(meth, metric)].mean() for meth in METHODS_TO_EVALUATE}
                   for metric in metrics})
df_std = pd.DataFrame({f'{metric} {directions[metric]}': {meth: results[(meth, metric)].std() for meth in METHODS_TO_EVALUATE}
                       for metric in metrics})
print('Mean over the images (± std):')
display((df.round(3).astype(str) + ' ± ' + df_std.round(3).astype(str)))

# Ranking per metric (1 = best), taking the direction of each metric into account
ranks = pd.DataFrame({c: df[c].rank(ascending=c.endswith('↓')) for c in df.columns})
fig, ax = plt.subplots(figsize=(2.6 * len(df.columns) + 2, 0.6 * len(df) + 1.6))
im = ax.imshow(ranks.values, cmap='RdYlGn_r', vmin=1, vmax=len(df))
ax.set_xticks(range(len(df.columns)), df.columns, rotation=20, ha='right'); ax.set_yticks(range(len(df)), df.index)
for i in range(len(df)):
    for j in range(len(df.columns)):
        ax.text(j, i, f'{df.values[i, j]:.2f}\n(rank {ranks.values[i, j]:.0f})', ha='center', va='center', fontsize=8)
ax.set_title('Quantus evaluation (rank 1 = best)'); plt.tight_layout(); plt.show()

In [ ]:
# Per-image scores for one (method, metric) pair
results[('Saliency', 'FaithfulnessCorrelation')]   # 🧠 change me

### 🧠 Exercise
Which method is best for each property? Is there a single winner? Now look at the ± std and at the per-image scores (cell above; change the key to see other method/metric pairs): how reliable are these conclusions?

*Hint:* Recall the trade-offs from the slides: smoothing usually helps robustness, coarse maps (Grad-CAM) are robust but may lose fine detail…

<details><summary>Answer</summary>

**IG** has the best faithfulness (≈ 0.22, vs. ≈ 0.17 for Grad-CAM and ≈ 0.02 for Saliency). **Grad-CAM** is by far the most robust (MaxSensitivity ≈ 0.25 vs. 1.5-2: a smooth 14×14 map barely changes under small noise) and the most class-sensitive (lowest RandomLogit similarity). IG is the least robust. So there is **no single winner**: each metric measures a different property. But the standard deviations (0.1-0.2) are as large as the differences in faithfulness, and the per-image scores of Saliency range from −0.27 to +0.18: with 4 images and tiny sample sizes these rankings are **not reliable**. Real evaluations use hundreds of images and also check the reliability of the metric itself (meta-evaluation, Hedström et al., 2023).

</details>

### 🧠 Exercise
Change `PERTURB_BASELINE` from `'black'` to `'uniform'` (and/or `NR_RUNS` from 20 to 50) and re-run the evaluation. Do the FaithfulnessCorrelation scores and the ranking change?

*Hint:* Faithfulness metrics *also* need a baseline to "remove" pixels, just like IG and Occlusion.

<details><summary>Answer</summary>

The FaithfulnessCorrelation values change noticeably: with `'uniform'`, Grad-CAM drops from ≈ 0.17 to ≈ −0.05 (from 2nd to last place) while IG stays ≈ 0.2; with `NR_RUNS = 50` the values shrink (IG ≈ 0.14, Grad-CAM ≈ 0.10). The metric depends on its own hyperparameters (what "removing" a pixel means, subset size, number of runs) and on random sampling. This is why evaluation results must report these choices, and why meta-evaluation checks metrics for **noise stability** and **sensitivity to hyperparameters**. Also note that replacing pixels with black/uniform values creates out-of-distribution inputs (cf. ROAD, which tries to mitigate this).

</details>

### 🧠 Exercise
Set `PERTURB_BASELINE` back to `'black'` and add `'Guided Backprop'` and `'SmoothGrad'` to `METHODS_TO_EVALUATE`. How does Guided Backprop score on robustness and on RandomLogit? Does this match the sanity check of §9? Does SmoothGrad improve over Saliency?

*Hint:* RandomLogit compares the explanation of the predicted class with the explanation of a random class (SSIM; 1 = identical).

<details><summary>Answer</summary>

Guided Backprop is very **robust** (MaxSensitivity ≈ 0.3, close to Grad-CAM) but its RandomLogit similarity is ≈ **1.0**: the explanation of a random class is identical to that of the predicted class, the same class-insensitivity we saw in §2 and in the randomisation test (§9). A method can thus score well on one property precisely *because* it ignores the model: robustness alone rewards explanations that do not depend on anything. SmoothGrad improves on Saliency in both robustness (≈ 1.2 vs 1.5) and faithfulness (≈ 0.10 vs 0.02). Always evaluate several complementary properties.

</details>

## 11. *(Optional)* Extras: `extra_attributions`

**This section is optional**, if you have time left. The package [`extra_attributions`](https://github.com/OscarPellicer/extra-attributions) (by the instructor) implements three methods with Captum's interface:

* `TiledOcclusion`: runs Occlusion several times on shifted grids (shift = `window / k`) and combines them, increasing the resolution by `k` without shrinking the occlusion window.
* `FusionGrad`: SmoothGrad on the inputs **plus** multiplicative noise on the **weights** (NoiseGrad), averaged.
* `ContrastiveAttribution`: explains `logit[target] − logit[negative_target]`, or `logit[target] − mean(other logits)` (the default).

In [ ]:
from extra_attributions import TiledOcclusion, FusionGrad, ContrastiveAttribution

TARGET = pred_idx    # 🧠 change me
K = 4                # 🧠 change me: resolution factor of TiledOcclusion (must divide the window, e.g. 1, 2, 4, 8)
WINDOW = 32

attr_occ = Occlusion(model).attribute(x, target=TARGET, sliding_window_shapes=(3, WINDOW, WINDOW),
                                      strides=(3, WINDOW, WINDOW), baselines=0, perturbations_per_eval=32)
attr_tocc = TiledOcclusion(model).attribute(x, target=TARGET, k=(1, K, K), window=(3, WINDOW, WINDOW),
                                            baselines=0, perturbations_per_eval=32, show_progress=False)
plot_attributions(x_img, [attr_occ, attr_tocc], [f'Occlusion (w=s={WINDOW})', f'TiledOcclusion (w={WINDOW}, k={K})'],
                  title=f'Target: {idx_to_label(TARGET)}')

In [ ]:
# FusionGrad on Saliency (we pass a copy of the model, since FusionGrad perturbs the weights in place)
fg_model = copy.deepcopy(model)
fusiongrad = FusionGrad(Saliency(fg_model), model=fg_model)
stdev = 0.15 * (x.max() - x.min()).item()
attr_ng = fusiongrad.attribute(x, target=TARGET, std=0.1, mean=1., n=8, additive_noise=False,  # weight noise only (NoiseGrad)
                               sg_std=0., m=1, sg_additive_noise=True, show_progress=False, abs=True)
attr_fg = fusiongrad.attribute(x, target=TARGET, std=0.1, mean=1., n=4, additive_noise=False,  # weight + input noise
                               sg_std=stdev, m=4, sg_additive_noise=True, show_progress=False, abs=True)
attr_sg = NoiseTunnel(Saliency(model)).attribute(x, nt_type='smoothgrad', nt_samples=16, nt_samples_batch_size=8,
                                                 stdevs=stdev, target=TARGET, abs=True)
plot_attributions(x_img, [Saliency(model).attribute(x, target=TARGET, abs=True), attr_sg, attr_ng, attr_fg],
                  ['|Saliency|', 'SmoothGrad', 'NoiseGrad', 'FusionGrad'], title=f'Target: {idx_to_label(TARGET)} (absolute gradients)')

In [ ]:
# ContrastiveAttribution (+ TiledOcclusion): target vs. one class, or vs. the average of all other classes
tocc_kw = dict(k=(1, 2, 2), window=(3, 32, 32), baselines=0, perturbations_per_eval=32, show_progress=False)
attr_a = TiledOcclusion(model).attribute(xc, target=a, **tocc_kw)
attr_vs_b = ContrastiveAttribution(TiledOcclusion(model)).attribute(xc, target=a, negative_target=b, **tocc_kw)
attr_vs_all = ContrastiveAttribution(TiledOcclusion(model)).attribute(xc, target=a, negative_target='other_classes', **tocc_kw)
plot_attributions(xc_img, [attr_a, attr_vs_b, attr_vs_all],
                  [f'TiledOcc: {CLASS_A}', f'{CLASS_A} − {CLASS_B}', f'{CLASS_A} − mean(others)'])

### 🧠 Exercise
In the TiledOcclusion cell, compare `K = 1`, `K = 2` and `K = 8`. What happens with `K = 1`? How does the cost grow?

*Hint:* `TiledOcclusion` runs `K × K` occlusions on grids shifted by `WINDOW / K` pixels.

<details><summary>Answer</summary>

With `K = 1` TiledOcclusion is identical to standard Occlusion with stride = window (7×7 blocks). Larger `K` gives a `K`-times finer map while keeping the 32-pixel occlusion window (so the occluded region is still large enough to change the prediction), at the cost of `K²` occlusion passes: the same cost as standard Occlusion with stride `WINDOW / K`. Note that TiledOcclusion *sums* the `K²` passes, so its values are ~`K²` times larger (see the colour bar).

</details>

## Take-home messages

* Different attribution methods answer **different questions** and can produce very different maps for the same prediction; their hyperparameters (baseline, `n_steps`, noise, layer, window, LRP rules) matter as much as the method.
* **Visually appealing ≠ faithful**: Guided Backprop gives beautiful maps that barely depend on the model (sanity check).
* **Contrastive** explanations (*why A rather than B?*) often answer the user's real question better than single-class ones.
* Quantitative evaluation (Quantus) is necessary, but metrics measure different properties, have hyperparameters of their own and need enough samples: always report the setup and the variability.

**References:** Simonyan et al. 2014 (Saliency) · Springenberg et al. 2015 (Guided Backprop) · Sundararajan et al. 2017 (IG) · Smilkov et al. 2017 (SmoothGrad) · Selvaraju et al. 2017 (Grad-CAM) · Zeiler & Fergus 2014 (Occlusion) · Bach et al. 2015, Montavon et al. 2019 (LRP) · Adebayo et al. 2018 (Sanity checks) · Bhatt et al. 2020, Yeh et al. 2019, Sixt et al. 2020 (metrics) · Hedström et al. 2023 (Quantus, meta-evaluation) · Bykov et al. 2022 (NoiseGrad/FusionGrad) · Miller 2019 (contrastive explanations).